In [ ]:
# Fine-tune BERT for text classification (Hugging Face)


In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score

model_name = "distilbert-base-uncased"
texts = ["I love this", "This is great", "I hate this", "This is terrible"] * 4
labels = [1, 1, 0, 0] * 4
dataset = Dataset.from_dict({"text": texts, "label": labels}).train_test_split(test_size=0.25, seed=42)

tokenizer = AutoTokenizer.from_pretrained(model_name)
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=64)
dataset = dataset.map(tokenize, batched=True)
dataset = dataset.remove_columns(["text"])
dataset.set_format("torch")

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
args = TrainingArguments(output_dir="bert_results", num_train_epochs=1, per_device_train_batch_size=4,
                         per_device_eval_batch_size=4, eval_strategy="epoch", report_to="none")
trainer = Trainer(model=model, args=args, train_dataset=dataset["train"], eval_dataset=dataset["test"])
trainer.train()
print(trainer.evaluate())
